In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, explode, map_keys, map_values,
    round as spark_round,lit, from_unixtime
)
from delta.tables import DeltaTable

In [0]:
DATA_PATH   = "/Volumes/main/default/telemetry/eventDate=2025-11-13/"  
BRONZE_PATH = "Volumes/main/default/telemetry/bronze_telemetry"
SILVER_PATH = "Volumes/main/default/telemetry/silver_telemetry"
GOLD_PATH   = "Volumes/main/default/telemetry/gold_telemetry"

DATABASE_NAME = "hive_streaming"

In [0]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}")
spark.sql(f"USE {DATABASE_NAME}")

DataFrame[]

In [0]:
# Read last processed date from checkpoint
checkpoint_df = spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DATABASE_NAME}.pipeline_checkpoint (
        last_processed_date DATE,
        last_run_timestamp  TIMESTAMP,
        rows_processed      LONG
    )
""") if spark.catalog.tableExists(f"{DATABASE_NAME}.pipeline_checkpoint") else None

last_processed_date = checkpoint_df.collect()[0][0] if checkpoint_df and checkpoint_df.count() > 0 else None

DataFrame[]

In [0]:
%sql
select * from hive_streaming.pipeline_checkpoint

last_processed_date,last_run_timestamp,rows_processed
2025-11-13,2026-05-30T18:18:53.636Z,566


In [0]:
df_source = spark.read.format("delta").load(DATA_PATH)

# Only read NEW data since last checkpoint
if last_processed_date:
    df_bronze = df_source.filter(col("eventDate") > lit(last_processed_date))
else:
    df_bronze = df_source  # first run — read everything

In [0]:
df_bronze.limit(10).display()

customerId,contentId,clientId,eventDate,timestampInfo,player,totalDistribution,qualityDistribution
4cb0f1c9-19b3-408c-8a09-03bf153819cf,5761d026-0569-437e-9033-16618b2c28fa,9359a34a-29de-4311-88a6-eebd158955ce,2025-11-13,"List(1763032282190, 1763032282169)","List(2, 8469)","List(List(9, 8.0, 14545184, 14545184), List(0, 0.0, 0, 0))","Map(720p -> List(List(2, 2.0, 5204968, 5204968), List(0, 0.0, 0, 0)), 270p -> List(List(2, 2.0, 984744, 984744), List(0, 0.0, 0, 0)))"
4cb0f1c9-19b3-408c-8a09-03bf153819cf,5761d026-0569-437e-9033-16618b2c28fa,22c0a830-c037-4a13-bb64-78d9c396df1f,2025-11-13,"List(1763032289079, 1763032288933)","List(2, 12673)","List(List(5, 4.0, 9215760, 9215760), List(1, 1.0, 325428, 325428))","Map(720p -> List(List(2, 2.0, 7942248, 7942248), List(1, 1.0, 325428, 325428)), 1080p -> List(List(1, 0.0, 0, 0), List(0, 0.0, 0, 0)), 270p -> List(List(2, 2.0, 1273512, 1273512), List(0, 0.0, 0, 0)))"
4cb0f1c9-19b3-408c-8a09-03bf153819cf,5761d026-0569-437e-9033-16618b2c28fa,6fc0c270-0ce5-40aa-88bd-2ed26cc27141,2025-11-13,"List(1763032283473, 1763032284712)","List(0, 0)","List(List(4, 2.0, 5156840, 5156840), List(2, 2.0, 629800, 629800))","Map(720p -> List(List(0, 0.0, 0, 0), List(1, 1.0, 229172, 229172)), 1080p -> List(List(1, 0.0, 0, 0), List(0, 0.0, 0, 0)), 270p -> List(List(1, 1.0, 575092, 575092), List(0, 0.0, 0, 0)))"
4cb0f1c9-19b3-408c-8a09-03bf153819cf,5761d026-0569-437e-9033-16618b2c28fa,ab7fff07-5941-428e-9bd5-e72ce89388a7,2025-11-13,"List(1763032288174, 1763032288147)","List(3, 16152)","List(List(4, 3.0, 13041372, 13041372), List(4, 3.0, 1379356, 1379356))","Map(720p -> List(List(1, 1.0, 3315380, 3315380), List(0, 0.0, 0, 0)), 1080p -> List(List(3, 2.0, 9725992, 9725992), List(0, 0.0, 0, 0)), 270p -> List(List(0, 0.0, 0, 0), List(4, 3.0, 1379356, 1379356)))"
4cb0f1c9-19b3-408c-8a09-03bf153819cf,5761d026-0569-437e-9033-16618b2c28fa,9ac455d2-7257-45a3-bd21-a6242a0c43e1,2025-11-13,"List(1763032288214, 1763032287559)","List(1, 26436)","List(List(2, 2.0, 9575592, 9575592), List(4, 4.0, 1629584, 1629584))","Map(720p -> List(List(1, 0.0, 0, 0), List(2, 2.0, 461352, 461352)), 1080p -> List(List(1, 2.0, 9575592, 9575592), List(0, 0.0, 0, 0)), 270p -> List(List(0, 0.0, 0, 0), List(2, 2.0, 1168232, 1168232)))"
4cb0f1c9-19b3-408c-8a09-03bf153819cf,5761d026-0569-437e-9033-16618b2c28fa,46c9a090-1115-402a-94d2-30b4ee1269fa,2025-11-13,"List(1763032288684, 1763032288587)","List(1, 3774)","List(List(4, 2.0, 5599016, 5599016), List(1, 1.0, 105844, 105844))","Map(1080p -> List(List(3, 1.0, 4900596, 4900596), List(0, 0.0, 0, 0)), 270p -> List(List(1, 1.0, 698420, 698420), List(1, 1.0, 105844, 105844)))"
4cb0f1c9-19b3-408c-8a09-03bf153819cf,5761d026-0569-437e-9033-16618b2c28fa,ca4c2a03-63de-4386-bf8b-ac0132fcb6df,2025-11-13,"List(1763032287463, 1763032285698)","List(0, 0)","List(List(5, 5.0, 9508100, 9508100), List(0, 0.0, 0, 0))","Map(720p -> List(List(5, 5.0, 9508100, 9508100), List(0, 0.0, 0, 0)))"
4cb0f1c9-19b3-408c-8a09-03bf153819cf,5761d026-0569-437e-9033-16618b2c28fa,6d26c8bb-4910-45fe-aabb-e20166be7ddb,2025-11-13,"List(1763032211604, 1763032211569)","List(0, 0)","List(List(1, 2.0, 4543208, 4543208), List(5, 5.0, 3732740, 3732740))","Map(720p -> List(List(1, 2.0, 4543208, 4543208), List(4, 4.0, 3115536, 3115536)))"
4cb0f1c9-19b3-408c-8a09-03bf153819cf,5761d026-0569-437e-9033-16618b2c28fa,e7d7517f-31a5-4892-9cf2-b221cdb2cb6f,2025-11-13,"List(1763032218338, 1763032218281)","List(2, 4942)","List(List(10, 9.0, 29477460, 29477460), List(1, 0.0, 0, 0))","Map(720p -> List(List(2, 2.0, 4543208, 4543208), List(0, 0.0, 0, 0)), 1080p -> List(List(6, 5.0, 23961540, 23961540), List(1, 0.0, 0, 0)), 270p -> List(List(2, 2.0, 972712, 972712), List(0, 0.0, 0, 0)))"
4cb0f1c9-19b3-408c-8a09-03bf153819cf,5761d026-0569-437e-9033-16618b2c28fa,01a63152-9513-4010-ac2e-d82af93a7391,2025-11-13,"List(1763032211762, 1763032211705)","List(3, 13850)","List(List(7, 6.0, 11545080, 11545080), List(2, 2.0, 1150184, 1150184))","Map(720p -> List(List(1, 1.0, 2241

In [0]:
(df_bronze
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(f"{DATABASE_NAME}.bronze_telemetry"))
    
print(f"Bronze — {df_bronze.count()} new records appended")

Bronze — 566 new records appended


Validating the raw data

In [0]:
# Row count
total_rows = df_bronze.count()
print(f"Total rows        : {total_rows}")
 
# Unique viewers, customers, content
n_viewers   = df_bronze.select("clientId").distinct().count()
n_customers = df_bronze.select("customerId").distinct().count()
n_content   = df_bronze.select("contentId").distinct().count()
 
print(f"Unique viewers    : {n_viewers}")
print(f"Unique customers  : {n_customers}")
print(f"Unique content    : {n_content}")

Total rows        : 566
Unique viewers    : 20
Unique customers  : 1
Unique content    : 1


In [0]:
# Null check on all columns
print("Null counts per column:")
df_bronze.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_bronze.columns
]).display()

Null counts per column:


customerId,contentId,clientId,eventDate,timestampInfo,player,totalDistribution,qualityDistribution
0,0,0,0,0,0,0,0


In [0]:
# Date range
print("Event dates found:")
df_bronze.select("eventDate").distinct().display()

Event dates found:


eventDate
2025-11-13


In [0]:
print("Records per viewer (min / max / avg):")
df_bronze.groupBy("clientId") \
    .count() \
    .agg(
        F.min("count").alias("min_records"),
        F.max("count").alias("max_records"),
        F.avg("count").alias("avg_records")
    ).display()

Records per viewer (min / max / avg):


min_records,max_records,avg_records
23,31,28.3


In [0]:
df_bronze.printSchema()

root
 |-- customerId: string (nullable = true)
 |-- contentId: string (nullable = true)
 |-- clientId: string (nullable = true)
 |-- eventDate: date (nullable = true)
 |-- timestampInfo: struct (nullable = true)
 |    |-- server: long (nullable = true)
 |    |-- agent: long (nullable = true)
 |-- player: struct (nullable = true)
 |    |-- bufferings: integer (nullable = true)
 |    |-- bufferingTime: integer (nullable = true)
 |-- totalDistribution: struct (nullable = true)
 |    |-- sourceTraffic: struct (nullable = true)
 |    |    |-- requests: integer (nullable = true)
 |    |    |-- responses: double (nullable = true)
 |    |    |-- requestedData: long (nullable = true)
 |    |    |-- receivedData: long (nullable = true)
 |    |-- p2pTraffic: struct (nullable = true)
 |    |    |-- requests: integer (nullable = true)
 |    |    |-- responses: double (nullable = true)
 |    |    |-- requestedData: long (nullable = true)
 |    |    |-- receivedData: long (nullable = true)
 |-- quali

Data cleaning for silver layer

In [0]:
# Flatten top-level nested structures
df_flat = df_bronze.select(
    col("customerId"),
    col("contentId"),
    col("clientId"),
    col("eventDate"),  # Timestamps: converted ms to seconds
    (col("timestampInfo.server") / 1000).cast("timestamp").alias("server_time"),
    (col("timestampInfo.agent")  / 1000).cast("timestamp").alias("agent_time"),
    col("player.bufferings").alias("bufferings"),  # Buffering
    spark_round(col("player.bufferingTime") / 1000, 2).alias("buffering_time_sec"),
    col("totalDistribution.sourceTraffic.requests").alias("cdn_requests"),# CDN (source) traffic
    col("totalDistribution.sourceTraffic.receivedData").alias("cdn_bytes"),
    col("totalDistribution.p2pTraffic.requests").alias("p2p_requests"),# P2P traffic
    col("totalDistribution.p2pTraffic.receivedData").alias("p2p_bytes"),
    col("qualityDistribution") 
)

In [0]:
# Explode qualityDistribution map into rows
# Each row becomes one quality level entry per viewer per report
df_quality_exploded = df_flat.select(
    col("customerId"),
    col("contentId"),
    col("clientId"),
    col("eventDate"),
    col("server_time"),
    col("agent_time"),
    col("bufferings"),
    col("buffering_time_sec"),
    col("cdn_requests"),
    col("cdn_bytes"),
    col("p2p_requests"),
    col("p2p_bytes"),
    explode(col("qualityDistribution")).alias("quality_level", "quality_data")
)

In [0]:
#  Flatten quality traffic
df_silver = df_quality_exploded.select(
    col("customerId"),
    col("contentId"),
    col("clientId"),
    col("eventDate"),
    col("server_time"),
    col("agent_time"),
    col("bufferings"),
    col("buffering_time_sec"),
    col("cdn_requests"),
    col("cdn_bytes"),
    col("p2p_requests"),
    col("p2p_bytes"),
    col("quality_level"),
    col("quality_data.sourceTraffic.receivedData").alias("quality_cdn_bytes"),
    col("quality_data.p2pTraffic.receivedData").alias("quality_p2p_bytes")
)

In [0]:
(df_silver
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(f"{DATABASE_NAME}.silver_telemetry"))

print(f"Silver table — {df_silver.count()} new records appended")

Silver table — 1259 new records appended


In [0]:
# Row count after explode 
silver_rows = df_silver.count()
print(f"Total rows after explode  : {silver_rows}")
print(f"Original bronze rows      : {total_rows}")
print(f"Ratio (quality levels/row): {round(silver_rows/total_rows, 2)}")

Total rows after explode  : 1259
Original bronze rows      : 566
Ratio (quality levels/row): 2.22


In [0]:
# Null check 
print("Null counts in silver")
df_silver.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_silver.columns
]).display()

Null counts in silver


customerId,contentId,clientId,eventDate,server_time,agent_time,bufferings,buffering_time_sec,cdn_requests,cdn_bytes,p2p_requests,p2p_bytes,quality_level,quality_cdn_bytes,quality_p2p_bytes
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
# Quality levels values
df_silver.select("quality_level").distinct().orderBy("quality_level").display()

quality_level
1080p
270p
360p
720p


In [0]:
# Buffering range 
print("Buffering stats")
df_silver.select(
    F.min("bufferings").alias("min_bufferings"),
    F.max("bufferings").alias("max_bufferings"),
    F.avg("bufferings").alias("avg_bufferings"),
    F.min("buffering_time_sec").alias("min_buffer_sec"),
    F.max("buffering_time_sec").alias("max_buffer_sec"),
    F.avg("buffering_time_sec").alias("avg_buffer_sec")
).display()

Buffering stats


min_bufferings,max_bufferings,avg_bufferings,min_buffer_sec,max_buffer_sec,avg_buffer_sec
0,5,1.374106433677522,0.0,37.08,4.800389197776013


In [0]:
df_silver.limit(5).display()

customerId,contentId,clientId,eventDate,server_time,agent_time,bufferings,buffering_time_sec,cdn_requests,cdn_bytes,p2p_requests,p2p_bytes,quality_level,quality_cdn_bytes,quality_p2p_bytes
4cb0f1c9-19b3-408c-8a09-03bf153819cf,5761d026-0569-437e-9033-16618b2c28fa,ab7fff07-5941-428e-9bd5-e72ce89388a7,2025-11-13,2025-11-13T11:08:57.559Z,2025-11-13T11:08:57.467Z,0,0.0,1,4756212,1,0,1080p,4756212,0
4cb0f1c9-19b3-408c-8a09-03bf153819cf,5761d026-0569-437e-9033-16618b2c28fa,ab7fff07-5941-428e-9bd5-e72ce89388a7,2025-11-13,2025-11-13T11:08:57.559Z,2025-11-13T11:08:57.467Z,0,0.0,1,4756212,1,0,270p,0,0
4cb0f1c9-19b3-408c-8a09-03bf153819cf,5761d026-0569-437e-9033-16618b2c28fa,22c0a830-c037-4a13-bb64-78d9c396df1f,2025-11-13,2025-11-13T11:08:54.235Z,2025-11-13T11:08:54.089Z,3,10.91,3,162996,7,15296056,360p,0,523956
4cb0f1c9-19b3-408c-8a09-03bf153819cf,5761d026-0569-437e-9033-16618b2c28fa,22c0a830-c037-4a13-bb64-78d9c396df1f,2025-11-13,2025-11-13T11:08:54.235Z,2025-11-13T11:08:54.089Z,3,10.91,3,162996,7,15296056,1080p,0,14271644
4cb0f1c9-19b3-408c-8a09-03bf153819cf,5761d026-0569-437e-9033-16618b2c28fa,22c0a830-c037-4a13-bb64-78d9c396df1f,2025-11-13,2025-11-13T11:08:54.235Z,2025-11-13T11:08:54.089Z,3,10.91,3,162996,7,15296056,270p,162996,500456


In [0]:
# Viewer Buffering Metrics
# One row per viewer with all buffering KPIs
 
df_gold_buffering = df_silver.groupBy(
    "customerId", "contentId", "clientId"
).agg(
    # Buffering metrics
    F.sum("bufferings").alias("total_bufferings"),
    spark_round(F.sum("buffering_time_sec"), 2).alias("total_buffering_time_sec"),
    spark_round(F.avg("buffering_time_sec"), 2).alias("avg_buffering_time_sec"),
    spark_round(F.max("buffering_time_sec"), 2).alias("max_buffering_time_sec"),
 
    # Traffic metrics
    F.sum("cdn_bytes").alias("total_cdn_bytes"),
    F.sum("p2p_bytes").alias("total_p2p_bytes"),
    F.sum(F.col("cdn_bytes") + F.col("p2p_bytes")).alias("total_bytes"),
 
    # Derived: P2P ratio (how much came from P2P vs total)
    spark_round(
        F.sum("p2p_bytes") / F.nullif(
            F.sum(F.col("cdn_bytes") + F.col("p2p_bytes")), F.lit(0)
        ) * 100, 2
    ).alias("p2p_percentage")
)

In [0]:
if spark.catalog.tableExists(f"{DATABASE_NAME}.gold_viewer_buffering"):
    gold_buffering_table = DeltaTable.forName(spark, f"{DATABASE_NAME}.gold_viewer_buffering")
    gold_buffering_table.alias("existing").merge(
        df_gold_buffering.alias("new"),
        """existing.customerId = new.customerId AND
           existing.clientId   = new.clientId   AND
           existing.contentId  = new.contentId"""
    ).whenMatchedUpdateAll()    \
     .whenNotMatchedInsertAll() \
     .execute()
    print("Gold buffering MERGE complete")

else:
    (df_gold_buffering
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{DATABASE_NAME}.gold_viewer_buffering"))
    print("Gold buffering table created")

Gold buffering MERGE complete


In [0]:
# Quality Distribution per Viewer 
# One row per viewer per quality level
 
df_gold_quality = df_silver.groupBy(
    "customerId", "contentId", "clientId", "quality_level"
).agg(
    F.sum("quality_cdn_bytes").alias("cdn_bytes"),
    F.sum("quality_p2p_bytes").alias("p2p_bytes"),
    F.sum(
        F.col("quality_cdn_bytes") + F.col("quality_p2p_bytes")
    ).alias("total_bytes")
)

In [0]:
if spark.catalog.tableExists(f"{DATABASE_NAME}.gold_quality_distribution"):
    gold_quality_table = DeltaTable.forName(spark, f"{DATABASE_NAME}.gold_quality_distribution")
    gold_quality_table.alias("existing").merge(
        df_gold_quality.alias("new"),
        """existing.customerId   = new.customerId   AND
           existing.clientId     = new.clientId     AND
           existing.contentId    = new.contentId    AND
           existing.quality_level= new.quality_level"""
    ).whenMatchedUpdateAll()    \
     .whenNotMatchedInsertAll() \
     .execute()
    print("Gold quality MERGE complete")

else:
    (df_gold_quality
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{DATABASE_NAME}.gold_quality_distribution"))
    print("Gold quality table created")

Gold quality MERGE complete


In [0]:
# Overall Event Summary
# One row for the whole event — useful for KPI cards on dashboard
 
df_gold_summary = df_silver.groupBy(
    "customerId", "contentId"
).agg(
    F.countDistinct("clientId").alias("total_viewers"),
    F.sum("bufferings").alias("total_bufferings"),
    spark_round(F.avg("buffering_time_sec"), 2).alias("avg_buffering_time_sec"),
    spark_round(F.sum("buffering_time_sec"), 2).alias("total_buffering_time_sec"),
    F.sum("cdn_bytes").alias("total_cdn_bytes"),
    F.sum("p2p_bytes").alias("total_p2p_bytes"),
    spark_round(
        F.sum("p2p_bytes") / F.nullif(
            F.sum(F.col("cdn_bytes") + F.col("p2p_bytes")), F.lit(0)
        ) * 100, 2
    ).alias("p2p_percentage")
)

In [0]:
if spark.catalog.tableExists(f"{DATABASE_NAME}.gold_event_summary"):
    gold_summary_table = DeltaTable.forName(spark, f"{DATABASE_NAME}.gold_event_summary")
    gold_summary_table.alias("existing").merge(
        df_gold_summary.alias("new"),
        """existing.customerId = new.customerId AND
           existing.contentId  = new.contentId"""
    ).whenMatchedUpdateAll()    \
     .whenNotMatchedInsertAll() \
     .execute()
    print("Gold summary MERGE complete")

else:
    (df_gold_summary
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{DATABASE_NAME}.gold_event_summary"))
    print("Gold summary table created")

Gold summary MERGE complete


In [0]:
# No of viewers present
gold_viewers = df_gold_buffering.count()
print(f"No of viewers in gold table : {gold_viewers}")
 
# Buffering metrics look sensible 
print("Buffering metrics summary")
df_gold_buffering.select(
    F.min("total_bufferings").alias("min"),
    F.max("total_bufferings").alias("max"),
    F.avg("total_bufferings").alias("avg"),
).display()

No of viewers in gold table : 20
Buffering metrics summary


min,max,avg
68,118,86.5


In [0]:
# No viewer has 0 total bytes (data integrity)
zero_bytes = df_gold_buffering.filter(F.col("total_bytes") == 0).count()
print(f"No of Viewers with zero total bytes : {zero_bytes} (expected: 0)")
 
# P2P percentage is between 0 and 100
invalid_p2p = df_gold_buffering.filter(
    (F.col("p2p_percentage") < 0) | (F.col("p2p_percentage") > 100)
).count()
print(f"Invalid P2P percentages : {invalid_p2p} (expected: 0)")


No of Viewers with zero total bytes : 0 (expected: 0)
Invalid P2P percentages : 0 (expected: 0)


In [0]:
# All 4 quality levels present 
print("Quality levels in gold quality table:")
df_gold_quality.select("quality_level").distinct().orderBy("quality_level").display()

Quality levels in gold quality table:


quality_level
1080p
270p
360p
720p


In [0]:
print("Top 5 viewers by buffering time:")
df_gold_buffering.select(
    "clientId",
    "total_bufferings",
    "total_buffering_time_sec",
    "avg_buffering_time_sec",
    "p2p_percentage"
).orderBy(F.desc("total_buffering_time_sec")).limit(5).display()
 

Top 5 viewers by buffering time:


clientId,total_bufferings,total_buffering_time_sec,avg_buffering_time_sec,p2p_percentage
35a1e10d-9cad-4839-b5a3-1e1ba0913d25,118,478.69,6.65,12.38
9ac455d2-7257-45a3-bd21-a6242a0c43e1,82,468.48,6.99,43.5
22c0a830-c037-4a13-bb64-78d9c396df1f,105,444.18,7.4,22.14
ab7fff07-5941-428e-9bd5-e72ce89388a7,91,415.74,5.86,13.42
01a63152-9513-4010-ac2e-d82af93a7391,98,396.95,6.11,9.69


In [0]:
print("Overall event summary:")
df_gold_summary.display()
 
print("Gold checks complete")
print()
print("=" * 60)
print("PIPELINE COMPLETE")
print("Tables created:")
print("  → hive_streaming.bronze_telemetry")
print("  → hive_streaming.silver_telemetry")
print("  → hive_streaming.gold_viewer_buffering")
print("  → hive_streaming.gold_quality_distribution")
print("  → hive_streaming.gold_event_summary")
print("=" * 60)

Overall event summary:


customerId,contentId,total_viewers,total_bufferings,avg_buffering_time_sec,total_buffering_time_sec,total_cdn_bytes,total_p2p_bytes,p2p_percentage
4cb0f1c9-19b3-408c-8a09-03bf153819cf,5761d026-0569-437e-9033-16618b2c28fa,20,1730,4.8,6043.69,10241002208,2243182536,17.97


Gold checks complete

PIPELINE COMPLETE
Tables created:
  → hive_streaming.bronze_telemetry
  → hive_streaming.silver_telemetry
  → hive_streaming.gold_viewer_buffering
  → hive_streaming.gold_quality_distribution
  → hive_streaming.gold_event_summary


In [0]:
# Check if there is actually any new data processed in this run
total_rows = df_bronze.count()

if total_rows > 0:
    # Safely find the latest eventDate we just processed
    max_date_row = df_bronze.select(F.max("eventDate")).collect()
    latest_date = max_date_row[0][0] if max_date_row and max_date_row[0][0] is not None else None
    
    if latest_date:
        # Insert new checkpoint record
        spark.sql(f"""
            INSERT INTO {DATABASE_NAME}.pipeline_checkpoint
            VALUES ('{latest_date}', current_timestamp(), {total_rows})
        """)
        print(f"Checkpoint updated — next run will process data after: {latest_date}")
    else:
        print("Warning: Data exists but 'eventDate' column values are completely null. Checkpoint skipped.")
else:
    print("Pipeline run completed: No new bronze data was found to process. Checkpoint unchanged.")

Checkpoint updated — next run will process data after: 2025-11-13


In [0]:
%sql
-- # ── Query 1: Buffering time per viewer (Bar Chart) ────────────────────────────

SELECT
    clientId,
    total_bufferings,
    total_buffering_time_sec,
    avg_buffering_time_sec
FROM hive_streaming.gold_viewer_buffering
ORDER BY total_buffering_time_sec DESC



clientId,total_bufferings,total_buffering_time_sec,avg_buffering_time_sec
35a1e10d-9cad-4839-b5a3-1e1ba0913d25,118,478.69,6.65
9ac455d2-7257-45a3-bd21-a6242a0c43e1,82,468.48,6.99
22c0a830-c037-4a13-bb64-78d9c396df1f,105,444.18,7.4
ab7fff07-5941-428e-9bd5-e72ce89388a7,91,415.74,5.86
01a63152-9513-4010-ac2e-d82af93a7391,98,396.95,6.11
6d26c8bb-4910-45fe-aabb-e20166be7ddb,73,363.89,6.38
b4d80be9-7376-41bb-bb2e-237878e6103d,101,358.04,4.97
ca4c2a03-63de-4386-bf8b-ac0132fcb6df,92,333.01,5.12
313aa0e6-04ea-471b-8ec4-b347043b5e00,77,326.87,5.54
f63c5309-d0c6-4643-b9b0-63306d06f3d3,78,290.92,4.93


In [0]:
%sql
-- # ── Query 2: P2P vs CDN traffic per viewer (Stacked Bar Chart) ────────────────
SELECT
    clientId,
    total_cdn_bytes   / 1e6 AS cdn_mb,
    total_p2p_bytes   / 1e6 AS p2p_mb,
    p2p_percentage
FROM hive_streaming.gold_viewer_buffering
ORDER BY p2p_percentage DESC


clientId,cdn_mb,p2p_mb,p2p_percentage
9ac455d2-7257-45a3-bd21-a6242a0c43e1,422.609712,325.307868,43.5
bf61c04b-3382-4c7c-8f46-3203d2668a58,456.12748,210.086804,31.53
b4d80be9-7376-41bb-bb2e-237878e6103d,538.783936,234.667992,30.34
8e39b554-af6d-4db2-a3fd-0e511ceeb904,199.203108,75.8674,27.58
22c0a830-c037-4a13-bb64-78d9c396df1f,392.711696,111.64098,22.14
6fc0c270-0ce5-40aa-88bd-2ed26cc27141,642.512936,179.906976,21.88
9a7a06ce-62c6-4e31-a15c-3d1e70620797,483.515884,113.386936,19.0
6e3d805b-409b-4b57-9113-6ba3c8b2d684,376.618896,82.714736,18.01
44d4aa0b-5fa2-474c-adfb-07544846b35e,692.349856,138.308028,16.65
9359a34a-29de-4311-88a6-eebd158955ce,604.718168,118.349008,16.37


In [0]:
%sql
-- ── Query 3: Quality distribution across all viewers (Pie Chart) ──────────────

SELECT
    quality_level,
    SUM(total_bytes) / 1e6 AS total_mb
FROM hive_streaming.gold_quality_distribution
GROUP BY quality_level
ORDER BY total_mb DESC


quality_level,total_mb
1080p,2818.005132
720p,1712.112052
270p,368.994932
360p,206.185052


In [0]:
%sql
-- # ── Query 4: Overall event KPIs (Counter / KPI Cards) ────────────────────────
SELECT
    total_viewers,
    total_bufferings,
    avg_buffering_time_sec,
    p2p_percentage
FROM hive_streaming.gold_event_summary

total_viewers,total_bufferings,avg_buffering_time_sec,p2p_percentage
20,1730,4.8,17.97


In [0]:
%sql
-- # ── Query 5: Worst performing viewers (Table) ─────────────────────────────────

SELECT
    clientId,
    total_bufferings,
    total_buffering_time_sec  AS total_buffer_sec,
    avg_buffering_time_sec    AS avg_buffer_sec,
    max_buffering_time_sec    AS worst_buffer_sec,
    p2p_percentage
FROM hive_streaming.gold_viewer_buffering
ORDER BY total_buffering_time_sec DESC
LIMIT 10

clientId,total_bufferings,total_buffer_sec,avg_buffer_sec,worst_buffer_sec,p2p_percentage
35a1e10d-9cad-4839-b5a3-1e1ba0913d25,118,478.69,6.65,21.01,12.38
9ac455d2-7257-45a3-bd21-a6242a0c43e1,82,468.48,6.99,37.08,43.5
22c0a830-c037-4a13-bb64-78d9c396df1f,105,444.18,7.4,18.47,22.14
ab7fff07-5941-428e-9bd5-e72ce89388a7,91,415.74,5.86,16.15,13.42
01a63152-9513-4010-ac2e-d82af93a7391,98,396.95,6.11,24.0,9.69
6d26c8bb-4910-45fe-aabb-e20166be7ddb,73,363.89,6.38,19.44,15.4
b4d80be9-7376-41bb-bb2e-237878e6103d,101,358.04,4.97,14.67,30.34
ca4c2a03-63de-4386-bf8b-ac0132fcb6df,92,333.01,5.12,13.72,13.4
313aa0e6-04ea-471b-8ec4-b347043b5e00,77,326.87,5.54,20.67,7.58
f63c5309-d0c6-4643-b9b0-63306d06f3d3,78,290.92,4.93,16.09,6.56
